In [1]:
import pandas as pd
# from IPython.display import display

In [2]:
pd.__version__

'2.2.3'

In [6]:
ticker_file = '/Users/liuda/Local/data/trading/data/stocks/ticker/history/AMZN.csv'

ticker_data = pd.read_csv(ticker_file)
ticker_data = ticker_data[ticker_data['Date'] >= '2024-10-01']
ticker_data['Date'] = pd.to_datetime(ticker_data['Date'])
ticker_data.set_index('Date', inplace=True)

ticker_data['qt'] = ticker_data.index.to_period('Q')
###############################################################################

# 添加季度信息
ticker_data['qt'] = ticker_data.index.to_period('Q')

# 计算每个季度的最大值、最小值，并计算“中点”值
quarter_stats = ticker_data.groupby('qt')['Adj Close'].agg(['max', 'min']).reset_index()
quarter_stats['median_price'] = 0.5 * (quarter_stats['max'].shift(1) + quarter_stats['min'].shift(1))

ticker_data = ticker_data.merge(quarter_stats[['qt', 'median_price']], on='qt', how='left')

# DEBUG
ticker_2024q4 = ticker_data[ticker_data['qt'] == '2024Q4']
print(ticker_2024q4['Adj Close'].max())
print(ticker_2024q4['Adj Close'].min())
print((ticker_2024q4['Adj Close'].max()+ticker_2024q4['Adj Close'].min()) / 2)

print(ticker_data[ticker_data['qt'] == '2025Q1'][['Adj Close', 'median_price']].iloc[0])

232.92999267578125
180.8000030517578
206.86499786376953
Adj Close       220.220001
median_price    206.864998
Name: 64, dtype: float64


In [12]:
ticker_file = '/Users/liuda/Local/data/trading/data/stocks/ticker/history/AMZN.csv'

ticker_data = pd.read_csv(ticker_file)
ticker_data = ticker_data[ticker_data['Date'] >= '2024-10-01']
ticker_data['Date'] = pd.to_datetime(ticker_data['Date'])
ticker_data.set_index('Date', inplace=True)

ticker_data['qt'] = ticker_data.index.to_period('Q')

quarter_max = ticker_data.groupby(['qt'])['Adj Close'].transform('max')
quarter_min = ticker_data.groupby(['qt'])['Adj Close'].transform('min')
quarter_median = 0.5 * (quarter_max + quarter_min)

ticker_data['median_price'] = quarter_median  # curr quarter median

quarter_median_df = ticker_data.groupby('qt')['median_price'].first().reset_index()
quarter_median_df['last_quarter_median_price'] = quarter_median_df['median_price'].shift(1)   # last quarter median
ticker_data = ticker_data.merge(quarter_median_df[['qt', 'last_quarter_median_price']], on='qt', how='left')
ticker_data.drop(columns=['median_price'], inplace=True)

# DEBUG
ticker_2024q4 = ticker_data[ticker_data['qt'] == '2024Q4']
print(ticker_2024q4['Adj Close'].max())
print(ticker_2024q4['Adj Close'].min())
print((ticker_2024q4['Adj Close'].max()+ticker_2024q4['Adj Close'].min()) / 2)

print(ticker_data[ticker_data['qt'] == '2025Q1'][['Adj Close', 'last_quarter_median_price']].iloc[0])

232.92999267578125
180.8000030517578
206.86499786376953
Adj Close                    220.220001
last_quarter_median_price    206.864998
Name: 64, dtype: float64


In [15]:
ticker_data.groupby(['qt'])['last_quarter_median_price']

In [16]:
ticker_data[['qt', 'Adj Close', 'last_quarter_median_price']][-3:]

,qt,Adj Close,last_quarter_median_price
67,2025Q1,222.110001,206.864998
68,2025Q1,222.130005,206.864998
69,2025Q1,218.940002,206.864998


In [14]:
import pandas_ta as ta
import yfinance as yf

supertrend = ta.supertrend(high=ticker_data['High'], low=ticker_data['Low'], close=ticker_data['Close'], length=10, multiplier=3.0)

supertrend


,SUPERT_10_3.0,SUPERTd_10_3.0,SUPERTl_10_3.0,SUPERTs_10_3.0
Date,,,,
2010-04-21,0.00000,1,NaN,NaN
2010-04-22,NaN,1,NaN,NaN
2010-04-23,NaN,1,NaN,NaN
2010-04-26,NaN,1,NaN,NaN
2010-04-27,NaN,1,NaN,NaN
...,...,...,...,...
2025-04-10,197.82672,-1,NaN,197.82672
2025-04-11,197.82672,-1,NaN,197.82672
2025-04-14,197.82672,-1,NaN,197.82672


In [20]:

first_values = ticker_data.groupby(ticker_data['Date'].year)['Adj Close'].first()


KeyError: 'Date'

In [31]:

# 合并并计算涨幅
ticker_data = ticker_data.merge(first_values.rename('First_Value'), 
             left_on=ticker_data['Date'].dt.year, 
             right_index=True)

In [32]:
ticker_data

,key_0,Date,Adj Close,Close,High,Low,Open,Volume,First_Value
0,2010,2010-04-21,7.321500,7.321500,7.450000,7.176000,7.258500,147488000,7.321500
1,2010,2010-04-22,7.504500,7.504500,7.554500,7.294000,7.350500,303316000,7.321500
2,2010,2010-04-23,7.181500,7.181500,7.454500,7.121000,7.269000,379504000,7.321500
3,2010,2010-04-26,7.355500,7.355500,7.386500,7.145000,7.160000,186366000,7.321500
4,2010,2010-04-27,7.101000,7.101000,7.322000,7.055500,7.277500,172780000,7.321500
...,...,...,...,...,...,...,...,...,...
3767,2025,2025-04-10,181.220001,181.220001,186.869995,175.850006,185.440002,68302000,220.220001
3768,2025,2025-04-11,184.869995,184.869995,185.860001,178.000000,179.929993,50594300,220.220001
3769,2025,2025-04-14,182.119995,182.119995,187.440002,179.229996,186.839996,48002500,220.220001
3770,2025,2025-04-15,179.589996,179.589996,182.350006,177.929993,181.410004,43642000,220.220001


In [ ]:
ticker_data['YTD_Return'] = ticker_data['Adj Close'] / ticker_data['First_Value'] - 1

In [ ]:
ticker_data['Adj High'] = ticker_data['High'] + ticker_data['Adj Close'] - ticker_data['Close']
ticker_data['Adj Low'] = ticker_data['Low'] + ticker_data['Adj Close'] - ticker_data['Close']
ticker_data['Adj Open'] = ticker_data['Open'] + ticker_data['Adj Close'] - ticker_data['Close']


In [ ]:
ticker_data.shape

(3772, 10)

In [ ]:
ticker_data = ticker_data.sort_values(by="Date").reset_index(drop=True)

# 计算technical指标
ticker_data['Date'] = pd.to_datetime(ticker_data['Date'])
ticker_data.set_index('Date', inplace=True)

In [ ]:
ticker_data.columns

Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Adj High',
       'Adj Low', 'Adj Open'],
      dtype='object')

In [ ]:
from pandas.tseries.holiday import MO

resampled_data = ticker_data.resample(
	'MS', label="left", closed='left'
	# , offset=pd.Timedelta(5, unit='D')
).agg(
{
	"Open": "first",
	"High": "max",
	"Low": "min",
	"Adj High": "max",
	"Adj Open": "first",
	"Adj Low": "min",
	"Adj Close": "last",
	"Volume": "sum",
}
).dropna(subset=['Open', 'Adj Close', 'High', 'Low'])
resampled_data


,Open,High,Low,Adj High,Adj Open,Adj Low,Adj Close,Volume
Date,,,,,,,,
2010-04-01,7.258500,7.554500,6.845500,7.554500,7.258500,6.845500,6.855000,1622714000
2010-05-01,6.860000,6.972000,5.876000,6.972000,6.860000,5.876000,6.273000,2818198000
2010-06-01,6.248500,6.457500,5.300500,6.457500,6.248500,5.300500,5.463000,2667570000
2010-07-01,5.445000,6.244000,5.290000,6.244000,5.445000,5.290000,5.894500,3506524000
2010-08-01,5.957500,6.540500,5.878500,6.540500,5.957500,5.878500,6.241500,2268396000
...,...,...,...,...,...,...,...,...
2024-12-01,209.960007,233.000000,209.509995,233.000000,209.960007,209.509995,219.389999,760891000
2025-01-01,222.029999,241.770004,216.199997,241.770004,222.029999,216.199997,237.679993,652629200
2025-02-01,234.059998,242.520004,204.160004,242.520004,234.059998,204.160004,212.279999,783613700


In [ ]:
resampled_data = ticker_data.resample(
	'W-MON', label="left"  , closed='left'
).agg(
	{
		"Open": "first",
		"High": "max",
		"Low": "min",
		"Adj High": "max",
		"Adj Open": "first",
		"Adj Low": "min",
		"Adj Close": "last",
		"Volume": "sum",
	}
).dropna(subset=['Open', 'Adj Close', 'High', 'Low'])

In [ ]:
resampled_data

,Open,High,Low,Adj High,Adj Open,Adj Low,Adj Close,Volume
Date,,,,,,,,
2010-04-19,7.258500,7.554500,7.121000,7.554500,7.258500,7.121000,7.181500,830308000
2010-04-26,7.160000,7.386500,6.845500,7.386500,7.160000,6.845500,6.855000,792406000
2010-05-03,6.860000,6.972000,6.030000,6.972000,6.860000,6.030000,6.249000,997942000
2010-05-10,6.486500,6.849500,6.338000,6.849500,6.486500,6.338000,6.426500,599454000
2010-05-17,6.412000,6.562500,5.876000,6.562500,6.412000,5.876000,6.136000,678474000
...,...,...,...,...,...,...,...,...
2025-03-17,198.770004,199.320007,189.380005,199.320007,198.770004,189.380005,196.210007,226177600
2025-03-24,200.000000,206.210007,191.880005,206.210007,200.000000,191.880005,192.720001,185517800
2025-03-31,188.190002,198.339996,166.000000,198.339996,188.190002,166.000000,171.000000,377207100


In [ ]:
df_test2 = resampled_data[-20:]

In [ ]:
df_test2[:-1]

,Open,High,Low,Adj High,Adj Open,Adj Low,Adj Close,Volume
Date,,,,,,,,
2023-09-01,139.460007,145.860001,123.040001,145.860001,139.460007,123.040001,127.120003,1120271900
2023-10-01,127.279999,134.479996,118.349998,134.479996,127.279999,118.349998,133.089996,1224564700
2023-11-01,133.960007,149.259995,133.710007,149.259995,133.960007,133.710007,146.089996,1025986900
2023-12-01,146.000000,155.630005,142.809998,155.630005,146.000000,142.809998,151.940002,931128600
2024-01-01,151.539993,161.729996,144.050003,161.729996,151.539993,144.050003,155.199997,953344900
2024-02-01,155.869995,177.220001,155.619995,177.220001,155.869995,155.619995,176.759995,1045061200
2024-03-01,176.750000,181.699997,171.470001,181.699997,176.750000,171.470001,180.380005,701928900
2024-04-01,180.789993,189.770004,166.320007,189.770004,180.789993,166.320007,175.000000,917021100
2024-05-01,181.639999,191.699997,173.869995,191.699997,181.639999,173.869995,176.440002,892301700


In [ ]:
df_test = ticker_data.merge(resampled_data, on='Date', how='left', suffixes=('', '_y'))

In [ ]:
df_test['weekday'] = df_test.index.weekday

In [5]:
df_test=df_test.ffill()
df_test[-4:]

NameError: name 'df_test' is not defined

In [ ]:
df_test['weekday'] = df_test.index.weekday
df_test.to_csv('/Users/liuda/Desktop/1.csv', index=True)

In [ ]:
ticker_file = '/Users/liuda/Local/data/trading/data/stocks/ticker/history_5m/AMZN.csv'

ticker_data = pd.read_csv(ticker_file)

In [ ]:
ticker_data['Adj High'] = ticker_data['High'] + ticker_data['Adj Close'] - ticker_data['Close']
ticker_data['Adj Low'] = ticker_data['Low'] + ticker_data['Adj Close'] - ticker_data['Close']
ticker_data['Adj Open'] = ticker_data['Open'] + ticker_data['Adj Close'] - ticker_data['Close']


In [ ]:
ticker_data = ticker_data.sort_values(by="Datetime").reset_index(drop=True)

# 计算technical指标
ticker_data['Datetime'] = pd.to_datetime(ticker_data['Datetime'])
ticker_data.set_index('Datetime', inplace=True)


In [ ]:
ticker_data.shape

(4680, 9)

In [ ]:
resampled_data = ticker_data.resample(
	# 'h', label="left", offset='1h'
	'10min', label='left', offset='0m'
).agg(
{
	"Open": "first",
	"High": "max",
	"Low": "min",
	"Adj High": "max",
	"Adj Open": "first",
	"Adj Low": "min",
	"Adj Close": "last",
	"Volume": "sum",
}
).dropna(subset=['Open', 'Adj Close', 'High', 'Low'])  # .reset_index(drop=False)

In [4]:
resampled_data[-3:]

NameError: name 'resampled_data' is not defined

In [ ]:
df_test = ticker_data.merge(resampled_data, on='Datetime', how='left', suffixes=('', '_y'))

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10000)
pd.set_option('display.width', 4000)

df_test[-20:].to_csv('/Users/liuda/Desktop/2.csv', index=True)